# Tram and metro service frequency
**Author**:  Sian Wilson <br>
**Date**:  22nd October 2026 <br>
**Dataset Scope**: TNDS (Traveline National Dataset), NaPTAN <br>
**Report Type**: Ad-hoc analysis <br>

## Purpose

This notebook measures how often trams, metros and Underground trains call at stations in Great Britain. It's the light rail companion to `railway_stations.ipynb`, which covers National Rail, and is part of identifying "well-connected" stations for the NPPF work, potentially leading to a new dataset on the Planning Data platform.

It:

1. Downloads the regional timetable zips (TransXChange XML) from the [Traveline National Dataset](https://www.travelinedata.org.uk/traveline-open-data/traveline-national-dataset/) FTP server.
2. Picks out the light rail networks by operator: Manchester Metrolink, Blackpool Tramway, Nottingham Express Transit, West Midlands Metro, Sheffield Supertram, London Trams, Docklands Light Railway, London Underground, Tyne and Wear Metro, Edinburgh Trams and Glasgow Subway.
3. Works out each journey's time at every stop on a chosen date.
4. Joins the stops to DfT [NaPTAN](https://naptan.api.dft.gov.uk/) tram and metro data (area code 940), grouping platforms into stations and adding station names and coordinates.
5. Counts departures per hour at a chosen station.

The parsing and counting functions live in `helpers.py` in this folder.

### Caveats

- **Networks are selected by operator, not by the TNDS mode tag.** Mode tags are unreliable: Nottingham's tram is tagged as bus, and heritage railways and airport shuttles are tagged as tram. A new network would need adding to `NETWORKS` in `helpers.py`.
- **Pick an ordinary date.** Bank holidays aren't applied, and planned closures change the service. For example, Metrolink's city centre was closed from 19 Oct to 1 Nov 2026, and part of Sheffield Supertram (Gleadless Townend to Halfway) is currently closed.
- **Count departures, not arrivals plus departures.** At a through stop, each tram both arrives and departs, so adding them counts it twice.
- **A station's count covers all its platforms and lines**, so it includes both directions.
- **Some journeys are published in more than one overlapping file.** Exact duplicates are removed.
- **TNDS is refreshed weekly**, so re-download the zips (`refresh=True`) for current timetables.


In [1]:
from ftplib import FTP
import os
import requests
import getpass
import csv
import gzip
import pandas as pd
import zipfile
import re
import xml.etree.ElementTree as ET
from datetime import date, timedelta
import re
import collections

from helpers import download_regions, all_tram_calls, load_light_rail_stations, add_stations, trams_per_hour

data_dir = "../../data/"
os.makedirs(data_dir, exist_ok=True)


## Data Import

In [ ]:
## You must register for an account at https://www.travelinedata.org.uk/traveline-open-data/traveline-national-dataset/
## Once registered, run this cell and
## enter your Traveline username and password

tnds_user = input("TNDS email: ")
tnds_password = getpass.getpass("TNDS password: ")

In [3]:
# Access the data via FTP
ftp = FTP("ftp.tnds.basemap.co.uk")
ftp.login(tnds_user, tnds_password)

ftp.dir()  # list what's available

drwxr-xr-x 1 ftp ftp           4096 Aug 04  2026 TNDSV2.5
-r--r--r-- 1 ftp ftp         466473 Oct 22 05:06 bodscomparisonreport.csv
-r--r--r-- 1 ftp ftp         444046 Oct 22 05:06 bodscomparisonreport_global.csv
-r--r--r-- 1 ftp ftp           9018 Oct 17 09:20 duplicateservicereport.csv
-r--r--r-- 1 ftp ftp       33702433 Oct 22 05:06 EA.zip
-r--r--r-- 1 ftp ftp      147972299 Oct 17 09:20 EM.zip
-r--r--r-- 1 ftp ftp         684154 Oct 17 09:20 iom.zip
-r--r--r-- 1 ftp ftp       95613094 Oct 22 05:06 L.zip
-r--r--r-- 1 ftp ftp          65949 Oct 22 05:06 log.txt
-r--r--r-- 1 ftp ftp       70465469 Oct 17 09:20 NE.zip
-r--r--r-- 1 ftp ftp      187089579 Oct 17 09:20 NW.zip
-r--r--r-- 1 ftp ftp      217669355 Oct 17 09:21 S.zip
-r--r--r-- 1 ftp ftp      214856711 Oct 22 05:06 SE.zip
-r--r--r-- 1 ftp ftp        2075438 Oct 22 05:07 servicereport.csv
-r--r--r-- 1 ftp ftp      172817722 Oct 17 09:21 SW.zip
-r--r--r-- 1 ftp ftp       50332969 Oct 17 09:21 W.zip
-r--r--r-- 1 ftp ftp      126

In [4]:
def ftp_download(filename):
    path = os.path.join(data_dir, filename)
    with open(path, "wb") as f:
        ftp.retrbinary(f"RETR {filename}", f.write)
    return path

report = pd.read_csv(ftp_download("servicereport.csv"), dtype=str)
report.head()

,RowId,RegionCode,RegionOperatorCode,ServiceCode,LineName,Description,StartDate,NationalOperatorCode,DataSource
0,1,EA,729HO,13-41B-_-y08-1,41B,Sudbury - Long Melford - Glemsford/Clare,2026-10-12,HEDO,LA
1,2,EA,729HO,13-46A-_-y08-1,46A,Sudbury - Long Melford - Gt Waldingfield,2026-10-12,HEDO,LA
2,3,EA,729HO,13-46B-_-y08-1,46B,Sudbury - Long Melford - Gt Waldingfield,2026-10-12,HEDO,LA
3,4,EA,729HO,13-49-_-y08-2,49,Sudbury Bus Station - Chaucer Rd - Tesco - Wes...,2026-10-12,HEDO,LA
4,5,EA,729HO,13-80A-_-y08-2,80A,Colchester - Boxted,2026-10-12,HEDO,LA


## Regional data

In [7]:
# Let's explore the North West's data
region = "NW"
download_regions(tnds_user, tnds_password, data_dir, regions=[region])
region_calls, region_failed = all_tram_calls("2026-11-10", data_dir, regions=[region])
region_calls = add_stations(region_calls, load_light_rail_stations())
region_calls

NW: {'Manchester Metrolink': 90, 'Blackpool Tramway': 1}


,region,network,service_code,line,journey,stop,stop_name,arrival,departure,station,station_name,latitude,longitude
0,NW,Manchester Metrolink,NW_04_METL_Blue Line_13,Blue Line,1,9400ZZMAAUL2,Ashton-under-Lyne (Manchester Metrolink),NaT,0 days 05:48:00,9400ZZMAAUL,Ashton-under-Lyne (Manchester Metrolink),53.490344,-2.098078
1,NW,Manchester Metrolink,NW_04_METL_Blue Line_13,Blue Line,1,9400ZZMAAWT2,Ashton West (Manchester Metrolink),0 days 05:51:00,0 days 05:51:00,9400ZZMAAWT,Ashton West (Manchester Metrolink),53.490244,-2.109743
2,NW,Manchester Metrolink,NW_04_METL_Blue Line_13,Blue Line,1,9400ZZMAAMO2,Ashton Moss (Manchester Metrolink),0 days 05:54:30,0 days 05:54:30,9400ZZMAAMO,Ashton Moss (Manchester Metrolink),53.483680,-2.121917
3,NW,Manchester Metrolink,NW_04_METL_Blue Line_13,Blue Line,1,9400ZZMAAUD2,Audenshaw (Manchester Metrolink),0 days 05:57:00,0 days 05:57:00,9400ZZMAAUD,Audenshaw (Manchester Metrolink),53.478205,-2.130957
4,NW,Manchester Metrolink,NW_04_METL_Blue Line_13,Blue Line,1,9400ZZMADRO2,Droylsden (Manchester Metrolink),0 days 06:00:00,0 days 06:00:00,9400ZZMADRO,Droylsden (Manchester Metrolink),53.479492,-2.144809
...,...,...,...,...,...,...,...,...,...,...,...,...,...
33094,NW,Blackpool Tramway,NW_05_BLAC_Tram_1,Tram,VJ279,9400ZZBPSLY2,Stanley Road,1 days 00:24:00,1 days 00:24:00,9400ZZBPSLY,Stanley Road (Blackpool Tramway),53.916557,-3.020587
33095,NW,Blackpool Tramway,NW_05_BLAC_Tram_1,Tram,VJ279,9400ZZBPFMW2,Fishermans Walk,1 days 00:25:00,1 days 00:25:00,9400ZZBPFMW,Fisherman's Walk (Blackpool Tramway),53.919686,-3.017374
33096,NW,Blackpool Tramway,NW_05_BLAC_Tram_1,Tram,VJ279,9400ZZBPPRT2,London Street,1 days 00:27:00,1 days 00:27:00,9400ZZBPPRT,London Street (Blackpool Tramway),53.922640,-3.011355
33097,NW,Blackpool Tramway,NW_05_BLAC_Tram_1,Tram,VJ279,9400ZZBPVTA2,Victoria Street,1 days 00:28:00,1 days 00:28:00,9400ZZBPVTA,Victoria Street (Blackpool Tramway),53.924918,-3.008638


In [8]:
# Stations per network in this region
region_calls.groupby("network")["station"].nunique()

network
Blackpool Tramway       40
Manchester Metrolink    99
Name: station, dtype: int64

In [9]:
# Group by a specific station to get trams per hour
trams_per_hour(region_calls, "9400ZZMASTP")  # St Peter's Square

hour
00:00    19
05:00    14
06:00    50
07:00    62
08:00    64
09:00    64
10:00    64
11:00    64
12:00    64
13:00    64
14:00    64
15:00    64
16:00    64
17:00    64
18:00    64
19:00    64
20:00    55
21:00    48
22:00    48
23:00    48
Name: departures, dtype: int64

## National data

Group all the regional data together into one file that can be explored

In [10]:
download_regions(tnds_user, tnds_password, data_dir)  # skips zips already downloaded; refresh=True to re-download
calls, failed = all_tram_calls("2026-11-17", data_dir)
calls = add_stations(calls, load_light_rail_stations())

EA: none
EM: {'Nottingham Express Transit': 1}
L: {'London Underground': 31, 'Docklands Light Railway': 4, 'London Trams': 2}
NE: {'Tyne and Wear Metro': 8}
NW: {'Manchester Metrolink': 90, 'Blackpool Tramway': 1}
S: {'Glasgow Subway': 1, 'Edinburgh Trams': 1}
SE: none
SW: none
W: none
WM: {'West Midlands Metro': 1}
Y: {'Sheffield Supertram': 17}


In [11]:
calls.head()

,region,network,service_code,line,journey,stop,stop_name,arrival,departure,station,station_name,latitude,longitude
0,EM,Nottingham Express Transit,NETTRAM,TRAM,1,9400ZZNOWKS1,Wilkinson Street Tram Stop,NaT,0 days 05:46:00,9400ZZNOWKS,Wilkinson Street Tram Stop,52.971974,-1.178056
1,EM,Nottingham Express Transit,NETTRAM,TRAM,1,9400ZZNOBFD1,Basford Tram Stop,0 days 05:48:00,0 days 05:48:00,9400ZZNOBFD,Basford Tram Stop,52.981668,-1.178572
2,EM,Nottingham Express Transit,NETTRAM,TRAM,1,9400ZZNODVL1,David Lane Tram Stop,0 days 05:49:00,0 days 05:49:00,9400ZZNODVL,David Lane Tram Stop,52.984875,-1.182161
3,EM,Nottingham Express Transit,NETTRAM,TRAM,1,9400ZZNOHBV1,Highbury Vale Tram Stop,0 days 05:51:00,0 days 05:51:00,9400ZZNOHBV,Highbury Vale Tram Stop,52.989693,-1.189980
4,EM,Nottingham Express Transit,NETTRAM,TRAM,1,9400ZZNOBWL1,Bulwell Tram Stop,0 days 05:53:00,0 days 05:53:00,9400ZZNOBWL,Bulwell Tram Stop,52.999670,-1.195188


In [12]:
trams_per_hour(calls, "9400ZZNOWKS")  # Wilkinson Street Tram Stop, labelled above in row 0

hour
00:00    12
05:00     2
06:00    18
07:00    30
08:00    32
09:00    32
10:00    28
11:00    24
12:00    24
13:00    24
14:00    24
15:00    31
16:00    32
17:00    32
18:00    32
19:00    28
20:00    24
21:00    19
22:00    16
23:00    17
Name: departures, dtype: int64